In [0]:
%python


from pyspark.sql.functions import broadcast , expr ,col ,sum ,current_timestamp,lit
from pyspark.sql.types import DecimalType,ShortType

class EodProcessStaging:

    def __init__(self):
        self.spark = spark

    def read_current_day_txn(self):

        return (
                self.spark.read.table("dev.spark_db.order_stg")
                                .where("as_of_dt = current_date()")
        )
    def total_qty_of_account(self , txn_df):

        txn_sum = ( txn_df.groupBy("account_number","asset_external_id","as_of_dt")
                        .agg(
                            sum(col("quantity")).alias("quantity")
                            )
        )
        return txn_sum
    
    def asset_data(self , asset_list):
        
        asset_df = (
                    self.spark.read.table("dev.spark_db.asset_rate").alias("as") 
                                    .join(broadcast(asset_list).alias("al"), 
                                          expr("""as.asset_external_id = al.asset_external_id
                                                            and 
                                                  al.as_of_dt = as.as_of_dt
                                             """))
                                    .select(
                                            "as.asset_external_id",
                                            "as.asset_rate",
                                            "as.as_of_dt"
                                           )
                )
        return asset_df
    
    def total_val_of_qty(self , txn_sum):

        asset_list = txn_sum.select("asset_external_id","as_of_dt").distinct()

        asset_rate = self.asset_data(asset_list)

        txn_data = ( txn_sum.alias("ts").join(broadcast(asset_rate).alias("ar"),
                                            expr("ts.asset_external_id = ar.asset_external_id"),
                                            "inner")
                            .select("ts.account_number",
                                    "ts.asset_external_id",
                                    "ts.as_of_dt",
                                    expr("ts.quantity as total_qty").cast(DecimalType(18,2)),
                                    expr("ts.quantity  * ar.asset_rate").cast(DecimalType(18,2)).alias("total_amt"),
                                    lit(1).cast(ShortType()).alias("eod_stg_ver"),
                                    current_timestamp().alias("eod_stg_dml"))
                                 )

                                     
        return txn_data
    

    def write_to_eod_stg(self , txn_data):

        txn_data.write.format("delta")\
                        .mode("append")\
                        .saveAsTable("dev.spark_db.eod_stg")

    def main(self):

        txn_df = self.read_current_day_txn()

        txn_sum = self.total_qty_of_account(txn_df)

        txn_data = self.total_val_of_qty(txn_sum)

        self.write_to_eod_stg(txn_data)

if __name__ =='__main__':
    eps = EodProcessStaging()
    eps.main()      

In [0]:
%sql
select * from dev.spark_db.eod_stg;